# Supervised Learning

Supervised learning is the most widely practiced form of machine learning. The algorithm learns from **labeled examples** — data where the correct answer is known — and generalizes to make predictions on new, unseen inputs. Whether predicting tomorrow's temperature, classifying an email as spam, or diagnosing a disease from medical scans, supervised learning provides the framework.

This notebook goes deep into how supervised learning works: the mathematical foundation, the major algorithms for regression and classification, how models learn by minimizing error, and the fundamental challenges of overfitting, underfitting, and generalization. By the end, you will understand not just which algorithm to use, but why it works and where it fails.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification, load_iris, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                              classification_report, confusion_matrix)

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. The Supervised Learning Framework

Given a training dataset of $n$ examples $\{(x_1, y_1), (x_2, y_2), \ldots, (x_n, y_n)\}$, supervised learning seeks a function $f$ that maps inputs $x$ to outputs $y$.

- $x_i$ is a **feature vector** — a list of attributes describing the $i$-th example.
- $y_i$ is the **label** or **target** — the known correct answer.
- $f$ is the **model** — the learned function we want to find.

The learning process has three stages:

1. **Choose a model family** — linear, polynomial, tree, neural network. This defines the shape of $f$.
2. **Define a loss function** — a measure of how wrong the predictions are.
3. **Optimize** — adjust the model's parameters to minimize the loss on training data.

$$\hat{\theta} = \arg\min_{\theta} \frac{1}{n} \sum_{i=1}^{n} L(f_\theta(x_i), y_i)$$

After training, the model is evaluated on **test data** — examples it has never seen — to measure how well it generalizes.

Supervised learning splits into two tasks based on the type of target:

- **Regression** — $y$ is continuous (price, temperature, probability).
- **Classification** — $y$ is discrete (spam/ham, digit 0–9, disease type).

---

## 2. Loss Functions

The loss function quantifies prediction error. Different tasks use different losses.

### Regression Losses

**Mean Squared Error (MSE):**

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Penalizes large errors heavily (squaring amplifies outliers). Differentiable and convex for linear models — easy to optimize.

**Mean Absolute Error (MAE):**

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

More robust to outliers than MSE because it does not square errors.

### Classification Losses

**Cross-Entropy Loss** (log loss) — measures the difference between predicted probability distribution and true label:

$$L = -\sum_{i} y_i \log(\hat{y}_i)$$

For binary classification with true label $y \in \{0, 1\}$ and predicted probability $\hat{p}$:

$$L = -\left[ y \log(\hat{p}) + (1 - y) \log(1 - \hat{p}) \right]$$

Cross-entropy heavily penalizes confident wrong predictions. It is the standard loss for logistic regression and neural network classifiers.

**Hinge Loss** — used by Support Vector Machines. Penalizes predictions on the wrong side of the decision boundary.

In [ ]:
# Visualizing loss functions
errors = np.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(errors, errors**2, "b-", linewidth=2)
axes[0].set_title("MSE: $L = e^2$")
axes[0].set_xlabel("Error (y - ŷ)")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].plot(errors, np.abs(errors), "g-", linewidth=2)
axes[1].set_title("MAE: $L = |e|$")
axes[1].set_xlabel("Error (y - ŷ)")
axes[1].axvline(0, color="gray", linestyle="--", alpha=0.5)

p = np.linspace(0.01, 0.99, 200)
axes[2].plot(p, -np.log(p), "r-", linewidth=2, label="y=1: $-\\log(\\hat{p})$")
axes[2].plot(p, -np.log(1 - p), "orange", linewidth=2, label="y=0: $-\\log(1-\\hat{p})$")
axes[2].set_title("Cross-Entropy Loss")
axes[2].set_xlabel("Predicted probability ŷ")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

---

## 3. Linear Regression

Linear regression is the simplest and most interpretable regression algorithm. It models the relationship between features and target as a linear function:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_d x_d = w_0 + \mathbf{w}^T \mathbf{x}$$

- $w_0$ is the **bias** (intercept) — the predicted value when all features are zero.
- $w_1, \ldots, w_d$ are **weights** — how much each feature contributes.
- A positive weight means increasing the feature increases the prediction; negative means decreasing.

### 3.1 How It Learns

The model finds weights that minimize MSE. For linear regression, this has a **closed-form solution** (no iterative optimization needed):

$$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

This is called the **normal equation**. In practice, numerical solvers (SVD, QR decomposition) are used instead of matrix inversion for stability.

### 3.2 Assumptions

Classical linear regression assumes:
- Linear relationship between features and target.
- Independent observations.
- Homoscedasticity (constant error variance).
- Normally distributed errors.

Violations do not always prevent useful predictions, but they affect confidence intervals and statistical inference.

### 3.3 Evaluation

- **R² (coefficient of determination)** — proportion of variance explained. R² = 1 is perfect; R² = 0 means the model is no better than predicting the mean.
- **RMSE** — $\sqrt{\text{MSE}}$, in the same units as the target.

In [ ]:
# Linear regression on diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LinearRegression().fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)

print(f"R² on test set: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print("\nFeature coefficients (standardized):")
for name, coef in zip(diabetes.feature_names, lr.coef_):
    print(f"  {name:15s}: {coef:+.4f}")

---

## 4. Polynomial and Regularized Regression

When the relationship between features and target is nonlinear, linear regression underfits. **Polynomial regression** adds powers and interaction terms:

$$\hat{y} = w_0 + w_1 x + w_2 x^2 + w_3 x^3 + \cdots$$

This is still linear in the parameters (weights) — the nonlinearity is in the features. A linear model with polynomial features can fit curves.

Higher-degree polynomials risk **overfitting** — fitting training noise instead of the true pattern. **Regularization** penalizes large weights to prevent this.

### 4.1 Ridge Regression (L2)

Adds a penalty proportional to the sum of squared weights:

$$\text{Loss} = \text{MSE} + \alpha \sum_{j} w_j^2$$

Shrinks weights toward zero but rarely to exactly zero. All features remain in the model.

### 4.2 Lasso Regression (L1)

Adds a penalty proportional to the absolute value of weights:

$$\text{Loss} = \text{MSE} + \alpha \sum_{j} |w_j|$$

Can shrink some weights to exactly zero — performing automatic feature selection.

$\alpha$ controls regularization strength. $\alpha = 0$ is ordinary linear regression. Larger $\alpha$ means more shrinkage.

In [ ]:
# Polynomial regression: underfitting vs overfitting
X_poly, y_poly = make_regression(n_samples=80, n_features=1, noise=8, random_state=42)
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_poly, y_poly, test_size=0.3, random_state=42)

X_plot = np.linspace(X_poly.min(), X_poly.max(), 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
degrees = [1, 4, 15]
titles = ["Degree 1 (underfit)", "Degree 4 (good fit)", "Degree 15 (overfit)"]

for ax, deg, title in zip(axes, degrees, titles):
    poly = PolynomialFeatures(degree=deg)
    X_tr_poly = poly.fit_transform(X_train_p)
    X_plot_poly = poly.transform(X_plot)

    model = LinearRegression().fit(X_tr_poly, y_train_p)
    ax.scatter(X_train_p, y_train_p, alpha=0.5, s=20, label="Train")
    ax.scatter(X_test_p, y_test_p, alpha=0.5, s=20, marker="x", label="Test")
    ax.plot(X_plot, model.predict(X_plot_poly), "r-", linewidth=2)
    test_r2 = r2_score(y_test_p, model.predict(poly.transform(X_test_p)))
    ax.set_title(f"{title}\nTest R² = {test_r2:.3f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---

## 5. Logistic Regression

Despite its name, logistic regression is a **classification** algorithm. It models the probability that an input belongs to a class using the sigmoid function:

$$\hat{p} = \sigma(\mathbf{w}^T \mathbf{x} + w_0) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + w_0)}}$$

The sigmoid squashes any real number into the range (0, 1), interpretable as probability. If $\hat{p} \geq 0.5$, predict class 1; otherwise predict class 0.

For multi-class problems, **softmax regression** (multinomial logistic regression) generalizes this to multiple classes.

### 5.1 Why Not Linear Regression for Classification?

Linear regression outputs unbounded values — predicting 1.5 or -0.3 for a binary problem makes no sense. Logistic regression constrains output to valid probabilities.

### 5.2 Decision Boundary

The decision boundary is where $\hat{p} = 0.5$, which occurs when $\mathbf{w}^T \mathbf{x} + w_0 = 0$. This is a **linear boundary** in feature space. Logistic regression can only separate classes with a straight line (or hyperplane). Nonlinear boundaries require feature engineering or different algorithms.

### 5.3 Strengths and Limitations

- Fast to train, interpretable coefficients, outputs probabilities.
- Works well when classes are linearly separable.
- Struggles with complex nonlinear boundaries without feature engineering.

In [ ]:
# Logistic regression on Iris (binary: setosa vs rest)
iris = load_iris()
X_iris = iris.data[:, :2]  # sepal length and width
y_iris = (iris.target == 0).astype(int)  # setosa vs not

X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

log_reg = LogisticRegression().fit(X_tr, y_tr)
acc = accuracy_score(y_te, log_reg.predict(X_te))

# Decision boundary visualization
xx, yy = np.meshgrid(np.linspace(X_iris[:, 0].min()-0.5, X_iris[:, 0].max()+0.5, 200),
                      np.linspace(X_iris[:, 1].min()-0.5, X_iris[:, 1].max()+0.5, 200))
Z = log_reg.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, Z, levels=20, cmap="RdBu", alpha=0.6)
plt.contour(xx, yy, Z, levels=[0.5], colors="black", linewidths=2)
plt.scatter(X_iris[y_iris==1, 0], X_iris[y_iris==1, 1], c="blue", label="Setosa", edgecolors="k")
plt.scatter(X_iris[y_iris==0, 0], X_iris[y_iris==0, 1], c="red", label="Not Setosa", edgecolors="k")
plt.xlabel("Sepal Length")
plt.ylabel("Sepal Width")
plt.title(f"Logistic Regression Decision Boundary (accuracy={acc:.2f})")
plt.legend()
plt.show()

---

## 6. K-Nearest Neighbors (KNN)

KNN is a simple, intuitive algorithm with no explicit training phase. To classify a new point, find the $K$ closest training examples and let them vote. To regress, average their target values.

**Classification:** majority vote among K neighbors.
**Regression:** mean (or weighted mean) of K neighbors' targets.

### 6.1 Distance Metric

Closeness is measured by **Euclidean distance**:

$$d(\mathbf{x}, \mathbf{x}') = \sqrt{\sum_{j=1}^{d} (x_j - x'_j)^2}$$

Other metrics: Manhattan distance, Minkowski, cosine similarity.

### 6.2 Choosing K

- **Small K** (e.g., 1) — sensitive to noise, complex decision boundary, overfitting.
- **Large K** — smoother boundary, underfitting, includes irrelevant neighbors.
- **Odd K** for binary classification avoids ties.

### 6.3 Strengths and Limitations

- No training time (lazy learner), works for both regression and classification, naturally handles multi-class.
- Slow prediction (must compute distance to all training points), sensitive to feature scale (requires normalization), curse of dimensionality.

In [ ]:
# KNN: effect of K on decision boundary
X_knn, y_knn = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_knn, y_knn, test_size=0.3, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, k in zip(axes, [1, 5, 25]):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    acc = accuracy_score(y_te, knn.predict(X_te))

    xx, yy = np.meshgrid(np.linspace(X_knn[:, 0].min()-1, X_knn[:, 0].max()+1, 150),
                          np.linspace(X_knn[:, 1].min()-1, X_knn[:, 1].max()+1, 150))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", edgecolors="k", s=15)
    ax.set_title(f"K={k}, accuracy={acc:.3f}")

plt.tight_layout()
plt.show()

---

## 7. Naive Bayes

Naive Bayes applies **Bayes' theorem** to classification:

$$P(y | \mathbf{x}) = \frac{P(\mathbf{x} | y) \cdot P(y)}{P(\mathbf{x})}$$

To classify, pick the class with the highest posterior probability:

$$\hat{y} = \arg\max_y \; P(y | \mathbf{x}) = \arg\max_y \; P(\mathbf{x} | y) \cdot P(y)$$

The **naive** assumption: features are conditionally independent given the class:

$$P(\mathbf{x} | y) = P(x_1 | y) \cdot P(x_2 | y) \cdots P(x_d | y)$$

This assumption is almost always wrong in practice, yet Naive Bayes often works surprisingly well — especially for text classification (spam filtering, sentiment analysis).

### Variants

- **Gaussian Naive Bayes** — assumes features follow a normal distribution within each class.
- **Multinomial Naive Bayes** — for count data (word frequencies).
- **Bernoulli Naive Bayes** — for binary features (word present/absent).

Naive Bayes is fast, handles high-dimensional data well, and needs relatively little training data.

In [ ]:
# Naive Bayes on Iris (multi-class)
X_full, y_full = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

nb = GaussianNB().fit(X_tr, y_tr)
y_pred_nb = nb.predict(X_te)

print(f"Accuracy: {accuracy_score(y_te, y_pred_nb):.4f}")
print("\nClassification Report:")
print(classification_report(y_te, y_pred_nb, target_names=iris.target_names))

---

## 8. Decision Trees

A decision tree makes predictions by asking a sequence of yes/no questions about features. Each internal node tests a feature; each branch represents an outcome; each leaf node holds a prediction.

```
              [Age < 30?]
             /           \
          Yes             No
          /                 \
  [Income > 50k?]     [Credit > 700?]
    /        \           /         \
 Approve   Deny      Approve      Deny
```

### 8.1 How Trees Split

At each node, the algorithm chooses the feature and threshold that best separates the classes (or reduces regression error). Common criteria:

- **Gini impurity** (classification) — measures how often a randomly chosen sample would be incorrectly classified.
- **Entropy / Information gain** (classification) — measures reduction in uncertainty after a split.
- **MSE** (regression) — reduction in mean squared error after a split.

### 8.2 Strengths

- Highly interpretable — the tree can be drawn and read by humans.
- Handles numerical and categorical features natively.
- No need for feature scaling.
- Captures nonlinear relationships and interactions automatically.

### 8.3 Weaknesses

- Prone to **overfitting** — deep trees memorize training data.
- Unstable — small data changes can produce very different trees.
- Greedy splitting may miss globally optimal structure.

Controlling `max_depth`, `min_samples_split`, and `min_samples_leaf` limits tree complexity.

In [ ]:
# Decision tree visualization
X_tr, X_te, y_tr, y_te = train_test_split(iris.data[:, :2], iris.target,
                                           test_size=0.3, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, depth, title in zip(axes, [2, None]):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_tr, y_tr)
    acc = accuracy_score(y_te, dt.predict(X_te))
    plot_tree(dt, feature_names=["Sepal L", "Sepal W"], class_names=iris.target_names,
              filled=True, ax=ax, fontsize=8)
    depth_label = f"depth={depth}" if depth else "unlimited depth"
    ax.set_title(f"{depth_label}, accuracy={acc:.3f}")

plt.tight_layout()
plt.show()

---

## 9. Support Vector Machines (SVM)

SVM finds the **optimal hyperplane** that separates classes with the maximum margin — the widest possible gap between the boundary and the nearest data points (support vectors).

For linearly separable data, the decision boundary is:

$$\mathbf{w}^T \mathbf{x} + b = 0$$

SVM maximizes the margin $\frac{2}{\|\mathbf{w}\|}$ subject to all points being correctly classified.

### 9.1 Kernel Trick

When data is not linearly separable, SVM uses **kernels** to map features into a higher-dimensional space where a linear separator exists. Common kernels:

- **Linear** — no transformation, original feature space.
- **Polynomial** — $(\mathbf{x}^T \mathbf{x}' + c)^d$.
- **RBF (Gaussian)** — $\exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$. Most popular; handles complex boundaries.

### 9.2 Soft Margin

Real data is rarely perfectly separable. **Soft margin SVM** allows some misclassifications, controlled by hyperparameter $C$. Large $C$ penalizes misclassifications heavily (risk overfitting); small $C$ allows more errors (smoother boundary).

### 9.3 When to Use SVM

Effective in high-dimensional spaces, memory-efficient (uses only support vectors), versatile with kernels. Less suited for very large datasets (training time scales poorly) and does not provide probability estimates natively.

In [ ]:
# SVM with different kernels
X_svm, y_svm = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, class_sep=0.8, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_svm, y_svm, test_size=0.3, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
kernels = ["linear", "poly", "rbf"]

for ax, kernel in zip(axes, kernels):
    svm = SVC(kernel=kernel, C=1.0).fit(X_tr, y_tr)
    acc = accuracy_score(y_te, svm.predict(X_te))

    xx, yy = np.meshgrid(np.linspace(X_svm[:, 0].min()-1, X_svm[:, 0].max()+1, 150),
                          np.linspace(X_svm[:, 1].min()-1, X_svm[:, 1].max()+1, 150))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", edgecolors="k", s=15)
    ax.set_title(f"SVM ({kernel}), acc={acc:.3f}")

plt.tight_layout()
plt.show()

---

## 10. Ensemble Methods

Ensemble methods combine multiple models to produce a stronger predictor than any individual model. The key insight: a group of diverse, moderately accurate models can outperform a single highly tuned model.

### 10.1 Bagging (Bootstrap Aggregating)

Train multiple models on random subsets of the data (with replacement), then average predictions (regression) or vote (classification).

**Random Forest** — bagging with decision trees. Each tree sees a random subset of data and a random subset of features. Reduces overfitting and variance compared to a single deep tree.

### 10.2 Boosting

Train models sequentially, where each new model focuses on correcting the errors of the previous ones.

**Gradient Boosting** — each new tree fits the residual errors of the ensemble so far. **XGBoost**, **LightGBM**, and **CatBoost** are highly optimized implementations widely used in competitions and industry.

### 10.3 Stacking

Train diverse base models, then train a meta-model on their predictions. The meta-model learns how to best combine the base models.

Ensemble methods — especially gradient boosting and random forests — consistently rank among the top performers on tabular data problems.

In [ ]:
# Comparing single tree vs ensemble on Iris
X_tr, X_te, y_tr, y_te = train_test_split(iris.data, iris.target, test_size=0.3, random_state=42)

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=200),
    "SVM (RBF)": SVC(kernel="rbf"),
}

print(f"{'Model':25s} {'Accuracy':>10s} {'CV Mean':>10s} {'CV Std':>8s}")
print("-" * 55)
for name, model in models.items():
    model.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model.predict(X_te))
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=5)
    print(f"{name:25s} {acc:10.4f} {cv_scores.mean():10.4f} {cv_scores.std():8.4f}")

---

## 11. Bias-Variance Tradeoff

Every model's prediction error decomposes into three components:

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

**Bias** — error from overly simplistic assumptions. A linear model trying to fit a sine wave has high bias. High-bias models **underfit** — they miss real patterns.

**Variance** — error from sensitivity to training data fluctuations. A deep decision tree that memorizes every training point has high variance. High-variance models **overfit** — they learn noise.

**Irreducible noise** — randomness inherent in the data that no model can eliminate.

The tradeoff: as model complexity increases, bias decreases but variance increases. The sweet spot minimizes total error.

| Model Complexity | Bias | Variance | Risk |
|-----------------|------|----------|------|
| Low (linear) | High | Low | Underfitting |
| Medium | Balanced | Balanced | Good generalization |
| High (deep tree) | Low | High | Overfitting |

In [ ]:
# Bias-variance visualization: model complexity vs error
complexity = np.linspace(0, 1, 100)
bias_sq = (1 - complexity) ** 2
variance = complexity ** 2 * 0.8
noise = np.full_like(complexity, 0.1)
total = bias_sq + variance + noise

plt.figure(figsize=(9, 5))
plt.plot(complexity, bias_sq, "b-", label="Bias²", linewidth=2)
plt.plot(complexity, variance, "r-", label="Variance", linewidth=2)
plt.plot(complexity, total, "k--", label="Total Error", linewidth=2)
plt.axvline(complexity[np.argmin(total)], color="green", linestyle=":", label="Optimal complexity")
plt.xlabel("Model Complexity →")
plt.ylabel("Error")
plt.title("Bias-Variance Tradeoff")
plt.legend()
plt.show()

---

## 12. Overfitting and Underfitting

**Underfitting** — the model is too simple to capture the underlying pattern. Training error and test error are both high. Solutions: use a more complex model, add features, reduce regularization.

**Overfitting** — the model memorizes training data including noise. Training error is low but test error is high. Solutions: more training data, regularization, simpler model, early stopping, dropout (for neural networks), pruning (for trees).

The gap between training and test performance is the diagnostic:

- Training ≈ Test → good fit.
- Training << Test → overfitting.
- Training ≈ Test, both high → underfitting.

**Regularization** is the primary defense against overfitting. Ridge, Lasso, tree depth limits, SVM margin parameter $C$, and early stopping all constrain model complexity to improve generalization.

In [ ]:
# Overfitting diagnostic: train vs test accuracy across tree depths
X_ov, y_ov = make_classification(n_samples=300, n_features=10, n_informative=5, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_ov, y_ov, test_size=0.3, random_state=42)

depths = range(1, 21)
train_accs, test_accs = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    train_accs.append(accuracy_score(y_tr, dt.predict(X_tr)))
    test_accs.append(accuracy_score(y_te, dt.predict(X_te)))

plt.figure(figsize=(9, 5))
plt.plot(depths, train_accs, "b-o", label="Train accuracy", markersize=4)
plt.plot(depths, test_accs, "r-o", label="Test accuracy", markersize=4)
plt.xlabel("Tree max_depth (complexity)")
plt.ylabel("Accuracy")
plt.title("Overfitting: train accuracy keeps rising, test accuracy peaks then drops")
plt.legend()
plt.show()

---

## 13. Choosing an Algorithm

No single algorithm wins on every problem. Practical guidelines:

| Situation | Good Starting Points |
|-----------|---------------------|
| Small dataset, need interpretability | Logistic regression, decision tree |
| Tabular data, accuracy matters most | Gradient boosting (XGBoost, LightGBM) |
| Text classification | Naive Bayes, logistic regression |
| Nonlinear boundaries, medium data | SVM with RBF kernel, random forest |
| Need probability estimates | Logistic regression, random forest |
| High-dimensional sparse data | Linear models with regularization |
| Large dataset, complex patterns | Gradient boosting, neural networks |

The best approach is to try multiple algorithms with proper cross-validation and compare. Start simple (logistic regression or a shallow tree), establish a baseline, then try more complex models. If complexity does not improve validation performance, the simpler model is preferable.

---

## 14. Summary

Supervised learning trains models on labeled data to predict outputs for new inputs. **Regression** algorithms (linear, polynomial, regularized) predict continuous values. **Classification** algorithms (logistic regression, KNN, Naive Bayes, decision trees, SVM) predict discrete categories.

All supervised algorithms follow the same framework: define a model, choose a loss function, optimize parameters on training data, evaluate on test data. **Ensemble methods** combine multiple models for stronger predictions. The **bias-variance tradeoff** governs model complexity — too simple underfits, too complex overfits.

Understanding these algorithms — their assumptions, strengths, and failure modes — enables informed model selection and debugging. Rigorous evaluation — measuring performance with the right metrics, validating with cross-validation, and ensuring the model generalizes to real-world data — is what turns a trained model into a trustworthy one.